<a href="https://colab.research.google.com/github/AyodejiOluwafemi/Google-Collab-for-Sales-Ops/blob/main/Ayodeji_Salau_SQL_TakeHome.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🚨 **IMPORTANT — READ THIS FIRST**

Before you do anything, **make a private copy of this notebook in your own Google Drive**.

If you edit this shared version, **your work will not be saved** and may be overwritten at any time.

**How to make your copy:**
1. Go to **File → Save a copy in Drive…**
2. Rename it to: `YourName_SQL_TakeHome.ipynb`
3. Work only in *your* copy


## 📝 **SQL + Python Take-Home Assessment Instructions**

Welcome! This assessment is designed to evaluate your SQL, Python, and analytical thinking skills.  
Please expand this section and read these instructions carefully before beginning.




### ⏱️ Time Limit

You have **100 minutes** to complete the assessment. We expect the assessment to take no more than 90 minutes, with a 10 minute buffer for submission.

- Your timer began when you submitted the **Start Test** form.
- Your timer ends when you submit the [Finish Test Form](https://docs.google.com/forms/d/e/1FAIpQLSfVXcrAbDRT2KLbNiCtNpDsRSAT63DTUqc5d6wRP-PmrM65uw/viewform?usp=dialog).
- The timestamps on these forms (recorded by Google) determine your official elapsed time.

**If your Finish timestamp is more than 100 minutes after your Start timestamp, your submission will not be accepted.** If you are not able to complete all 3 questions and submit in 100 minutes, please just submit whatever you have completed.

---

### 📄 How to Submit

1. Go to **File → Save a copy in Drive…**  
2. Rename your copy to: YourName_SQL_TakeHome.ipynb
3. Complete the assessment
4. Submit the [Finish Test Form](https://docs.google.com/forms/d/e/1FAIpQLSfVXcrAbDRT2KLbNiCtNpDsRSAT63DTUqc5d6wRP-PmrM65uw/viewform?usp=dialog).

---
### Other Notes

⚠️ **Before answering any questions, scroll down and run the entire “📦 Data Setup & Environment Initialization” section.**

If your runtime resets (common in Colab), you must re-run Setup.


## 📚 **Data Dictionary**

Expand this section to see a reference of all datasets used in this assessment. These tables are created during the Setup phase and loaded into a SQLite in-memory database.



🏥 facilities table

Facility-level metadata for California skilled nursing facilities.

**Columns**

- `cms_id` — CMS Certification Number (primary key, uppercase string)  
- `name` — Facility name  
- `chain_name` — Corporate chain name (may be null)  
- `city` — City  
- `state` — State code ("IL")  
- `zip` — ZIP code (string)  
- `beds` — Number of certified beds  
- `rating` — CMS 5-star rating  
- `latitude` — Latitude  
- `longitude` — Longitude  

**Notes**

- Rows with missing `cms_id` removed  
- Duplicate `cms_id` removed  
- `cms_id` standardized to uppercase  

---

👨‍⚕️ pbj_hours table

Payroll-Based Journal (PBJ) staffing hours for RN, LPN, and CNA roles.

**Columns**

- `id` — Auto-increment primary key  
- `cms_id` — Facility CCN (foreign key to facilities)  
- `work_date` — PBJ entry date (YYYY-MM-DD)  
- `job_title` — RN, LPN, or CNA  
- `total_hours` — Hours worked for that role on that date  

**Notes**

- RN/LPN/CNA columns were melted into long format  
- Only rows with `total_hours > 0` included  
- `cms_id` standardized to uppercase  

---

🧑‍💼 admin_details table

Administrator contact information from state licensure data.

**Columns**

- `facname` — Raw facility name (may differ from CMS naming)  
- `address` — Street address  
- `city` — City  
- `zip` — ZIP code  
- `facadmin` — Administrator name  
- `contact_email` — Administrator email  

**Notes**

- Only active SNFs included  
- Columns normalized to lowercase  
- Some inconsistencies intentionally preserved for matching  

---

🕒 shifts table

Synthetic shift-level operational and financial data.

**Columns**

- `shift_id` — Primary key  
- `cms_id` — Facility CCN  
- `date` — Shift date (YYYY-MM-DD)  
- `specialty` — CNA, LPN, or RN  
- `hours` — Shift length (8 or 12)  
- `pay_rate` — Hourly pay rate  
- `charge_rate` — Hourly bill rate (pay_rate + markup)  

**Notes**

- 5–15 shifts generated per facility  
- `charge_rate` added during Setup  
- Used for revenue and commission calculations  

---

🤝 deals table

Rep attribution metadata for SNF deals.

**Columns**

- `deal_id` — Primary key  
- `cms_id` — Facility CCN  
- `rep_primary` — Primary rep  
- `rep_secondary` — Secondary rep (nullable)  
- `split_primary_pct` — Attribution percentage for primary rep  
- `split_secondary_pct` — Attribution percentage for secondary rep  

**Notes**

- Percentages sum to 1.0  
- Deals may be single-rep or split  

---

🛠️ Helper Function

`run_query(sql_string)` executes an SQL query against the in-memory database and returns a pandas DataFrame.

**Example**

```
run_query("""
SELECT *
FROM facilities
LIMIT 5
""")
```

---


## 📦 **Data Setup & Environment Initialization**

Do not modify the code under this section, but ensure it runs (hit the Run all button at the top under your toolbar before completing anything in the assessment section below)


In [ ]:
# ============================================================
# 📘 Test Setup (DO NOT MODIFY THIS CELL)
# ============================================================

!pip install gdown --quiet

import gdown
import sqlite3
import pandas as pd
import random, datetime

# ============================================================
# 🔗 STEP 0 — Download source CSVs using Google Drive file IDs
# ============================================================

SNF_FILE_ID = "1UfCxgMxUtCEDWqcm1udnd7mPawDh7y-b"
PBJ_FILE_ID = "1y9WofLddBZ7ufuAeJ0HEfW9uRlvuQTt7"
ADMIN_FILE_ID = "1mR7vOR3xyeZ6sv4QiclCftOYqB79bajT"

gdown.download(f"https://drive.google.com/uc?id={SNF_FILE_ID}", "IL_SNFs.csv", quiet=False)
gdown.download(f"https://drive.google.com/uc?id={PBJ_FILE_ID}", "IL_PBJ_Hours.csv", quiet=False)
gdown.download(f"https://drive.google.com/uc?id={ADMIN_FILE_ID}", "admin_details_raw.csv", quiet=False)

# ============================================================
# 📁 STEP 1 — Load SNF Facilities dataset
# ============================================================

facilities_df = pd.read_csv("IL_SNFs.csv")
facilities_df = facilities_df.rename(columns={
    "CMS Certification Number (CCN)": "cms_id",
    "Provider Name": "name",
    "City/Town": "city",
    "State": "state",
    "ZIP Code": "zip",
    "Number of Certified Beds": "beds",
    "Overall Rating": "rating",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Chain Name": "chain_name"
})
facilities_df = facilities_df[
    ["cms_id", "name", "city", "state", "zip", "beds", "rating", "latitude", "longitude", "chain_name"]
].copy()
facilities_df = facilities_df.dropna(subset=["cms_id"]).drop_duplicates(subset=["cms_id"])
facilities_df["cms_id"] = facilities_df["cms_id"].astype(str).str.strip().str.upper()

# ============================================================
# 🏥 STEP 2 — Load PBJ dataset (FIXED: zero-pad CCNs)
# ============================================================

pbj_df = pd.read_csv("IL_PBJ_Hours.csv")
pbj_df.columns = [c.strip().lower() for c in pbj_df.columns]
pbj_df = pbj_df[["provnum", "workdate", "hrs_rn_ctr", "hrs_lpn_ctr", "hrs_cna_ctr"]].copy()

# 🔥 CRITICAL FIX — zero-pad CCNs so they match facilities_df
pbj_df["provnum"] = (
    pbj_df["provnum"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.zfill(6)        # <—— prevents the top–100 mismatch bug
)

pbj_long_df = pbj_df.melt(
    id_vars=["provnum", "workdate"],
    value_vars=["hrs_rn_ctr", "hrs_lpn_ctr", "hrs_cna_ctr"],
    var_name="job_title",
    value_name="total_hours"
)

pbj_long_df["job_title"] = (
    pbj_long_df["job_title"]
    .str.replace("hrs_", "", regex=False)
    .str.replace("_ctr", "", regex=False)
    .str.upper()
)

pbj_long_df = pbj_long_df.dropna(subset=["total_hours"])
pbj_long_df = pbj_long_df[pbj_long_df["total_hours"] > 0]

pbj_long_df = pbj_long_df.rename(columns={
    "provnum": "cms_id",
    "workdate": "work_date"
})

# ============================================================
# 👤 STEP 3 — Load Admin Details (new dataset)
# ============================================================

admin_details_raw_df = pd.read_csv("admin_details_raw.csv")
admin_details_df = admin_details_raw_df[
    (admin_details_raw_df["FAC_TYPE_CODE"] == "SNF") &
    (admin_details_raw_df["LICENSE_STATUS_DESCRIPTION"] == "ACTIVE")
][["FACNAME", "ADDRESS", "CITY", "ZIP", "FACADMIN", "CONTACT_EMAIL"]].copy()
admin_details_df.columns = [c.lower() for c in admin_details_df.columns]

# ============================================================
# 🗄️ STEP 4 — Create SQLite in-memory DB
# ============================================================

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

cur.executescript("""
DROP TABLE IF EXISTS facilities;
CREATE TABLE facilities (
  cms_id TEXT PRIMARY KEY,
  name TEXT,
  chain_name TEXT,
  city TEXT,
  state TEXT,
  zip TEXT,
  beds INTEGER,
  rating REAL,
  latitude REAL,
  longitude REAL
);

DROP TABLE IF EXISTS pbj_hours;
CREATE TABLE pbj_hours (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  cms_id TEXT,
  work_date TEXT,
  job_title TEXT,
  total_hours REAL,
  FOREIGN KEY(cms_id) REFERENCES facilities(cms_id)
);

DROP TABLE IF EXISTS admin_details;
CREATE TABLE admin_details (
  facname TEXT,
  address TEXT,
  city TEXT,
  zip TEXT,
  facadmin TEXT,
  contact_email TEXT
);
""")

# ============================================================
# 📝 STEP 5 — Insert datasets into SQL
# ============================================================

facilities_df.to_sql("facilities", conn, if_exists="append", index=False)
pbj_long_df.to_sql("pbj_hours", conn, if_exists="append", index=False)
admin_details_df.to_sql("admin_details", conn, if_exists="append", index=False)

# ============================================================
# 🛠️ STEP 6 — Generate deterministic shifts table
# ============================================================

cur.executescript("""
DROP TABLE IF EXISTS shifts;
CREATE TABLE shifts (
  shift_id INTEGER PRIMARY KEY,
  cms_id TEXT,
  date TEXT,
  specialty TEXT,
  hours REAL,
  pay_rate REAL,
  FOREIGN KEY(cms_id) REFERENCES facilities(cms_id)
);
""")

random.seed(42)
specialties = ["CNA", "LPN", "RN"]
hours_options = [8, 12]
date_start = datetime.date(2025, 10, 1)

rows = []
shift_id = 1
for cms_id in facilities_df["cms_id"]:
    for _ in range(random.randint(5, 15)):
        rows.append((
            shift_id,
            cms_id,
            (date_start + datetime.timedelta(days=random.randint(0, 30))).isoformat(),
            random.choice(specialties),
            random.choice(hours_options),
            random.randint(30, 60)
        ))
        shift_id += 1

shifts_df = pd.DataFrame(rows, columns=[
    "shift_id", "cms_id", "date", "specialty", "hours", "pay_rate"
])
shifts_df.to_sql("shifts", conn, if_exists="append", index=False)

# ============================================================
# ⚡ STEP 7 — SQL helper
# ============================================================

def run_query(query: str):
    return pd.read_sql_query(query, conn)

# ============================================================
# 🆕 STEP 8 — Generate Deals Table
# ============================================================

shifts_df["charge_rate"] = shifts_df["pay_rate"] + shifts_df["pay_rate"].apply(
    lambda x: random.randint(15, 40)
)
cur.executescript("DROP TABLE IF EXISTS shifts;")
shifts_df.to_sql("shifts", conn, index=False, if_exists="replace")

cur.executescript("""
DROP TABLE IF EXISTS deals;
CREATE TABLE deals (
  deal_id INTEGER PRIMARY KEY,
  cms_id TEXT,
  rep_primary TEXT,
  rep_secondary TEXT,
  split_primary_pct REAL,
  split_secondary_pct REAL
);
""")

reps = ["Alex", "Taylor", "Jordan", "Morgan", "Casey"]
sample_facilities = random.sample(list(facilities_df["cms_id"]), 10)

deal_rows = []
deal_id = 1
for cms in sample_facilities:
    rep1 = random.choice(reps)
    if random.random() < 0.5:
        rep2 = random.choice([r for r in reps if r != rep1])
        p1, p2 = random.choice([(0.5, 0.5), (0.75, 0.25)])
    else:
        rep2 = None
        p1, p2 = 1.0, 0.0
    deal_rows.append((deal_id, cms, rep1, rep2, p1, p2))
    deal_id += 1

deal_df = pd.DataFrame(deal_rows, columns=[
    "deal_id", "cms_id", "rep_primary", "rep_secondary",
    "split_primary_pct", "split_secondary_pct"
])
deal_df.to_sql("deals", conn, index=False, if_exists="append")

print("Setup complete.")

Downloading...
From: https://drive.google.com/uc?id=1UfCxgMxUtCEDWqcm1udnd7mPawDh7y-b
To: /content/IL_SNFs.csv
100%|██████████| 726k/726k [00:00<00:00, 9.02MB/s]
Downloading...
From: https://drive.google.com/uc?id=1y9WofLddBZ7ufuAeJ0HEfW9uRlvuQTt7
To: /content/IL_PBJ_Hours.csv
100%|██████████| 17.1M/17.1M [00:00<00:00, 45.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1mR7vOR3xyeZ6sv4QiclCftOYqB79bajT
To: /content/admin_details_raw.csv
100%|██████████| 7.42M/7.42M [00:00<00:00, 31.4MB/s]
/tmp/ipykernel_19620/149852213.py:51: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  pbj_df = pd.read_csv("IL_PBJ_Hours.csv")


Setup complete.


## 🧪 **Assessment**

See below each of the 3 questions, with boxes for you to input your solution to each one.

📌 **Question 1 — PBJ Hours for the Top 10 Chains**

Create a SQL query that returns PBJ staffing activity for the **top 10 chains** in the dataset.

Your output **must include the following columns:**

- **chain_name** – Name of the chain  
- **total_facilities** – Total number of facilities that belong to that chain  
- **facilities_with_pbj_hours** – Number of chain facilities that reported > 0 hours in PBJ  
- **total_agency_hours** – Total CNA + LPN + RN hours for all facilities within that chain  
- **pct_of_statewide_hours** – Percentage of all PBJ hours in the state represented by this chain  

---

**Rules & Notes**

- Include all worker types (**RN, LPN, CNA**).  
- Only include PBJ entries with `total_hours > 0`.  
- Only include chains that appear in the **top 10 by total agency hours**.  
- Do **not** include rows where `chain_name` is NULL or blank.  
- Sort the result by **total_agency_hours DESC**.  


In [ ]:
# QUESTION 1
# Write your SQL query below
# Reminder that SQL must be wrapped in run_query() - example below

run_query("""
WITH ChainAggregates AS (
    -- 1. Combine facility metadata with pbj_hours
    SELECT
        f.chain_name,
        f.cms_id,
        p.total_hours
    FROM facilities f
    JOIN pbj_hours p ON f.cms_id = p.cms_id
    WHERE f.chain_name IS NOT NULL
      AND f.chain_name != ''
      AND p.total_hours > 0
),
StatewideTotal AS (
    -- 2. Get the denominator for the statewide percentage
    SELECT SUM(total_hours) as state_total
    FROM pbj_hours
    WHERE total_hours > 0
)
SELECT
    ca.chain_name,
    -- Count distinct facilities in the chain from the metadata table
    (SELECT COUNT(DISTINCT cms_id) FROM facilities WHERE chain_name = ca.chain_name) AS total_facilities,
    -- Count distinct facilities that actually reported hours
    COUNT(DISTINCT ca.cms_id) AS facilities_with_pbj_hours,
    -- Sum of all agency hours (CNA + LPN + RN)
    SUM(ca.total_hours) AS total_agency_hours,
    -- Percentage of total statewide hours
    (SUM(ca.total_hours) * 100.0 / (SELECT state_total FROM StatewideTotal)) AS pct_of_statewide_hours
FROM ChainAggregates ca
GROUP BY ca.chain_name
ORDER BY total_agency_hours DESC
LIMIT 10
""")

,chain_name,total_facilities,facilities_with_pbj_hours,total_agency_hours,pct_of_statewide_hours
0,PACS GROUP,125,84,172739.87,14.984653
1,MARINER HEALTH CARE,18,17,160999.32,13.966196
2,ASPEN SKILLED HEALTHCARE,30,22,49993.21,4.336757
3,BRIUS MANAGEMENT,36,18,47877.09,4.153190
4,ROLLINS-NELSON HEALTHCARE MANAGEMENT,10,7,45537.63,3.950250
5,GOLDEN SNF OPERATIONS,7,7,44516.85,3.861700
6,THE ENSIGN GROUP,69,34,39047.36,3.387239
7,EVA CARE GROUP,9,8,31544.50,2.736388
8,PROMEDICA SENIOR CARE,6,5,17417.54,1.510918
9,LINKS HEALTHCARE GROUP,21,11,16445.25,1.426575


### 📌 **Question 2 — Match Admin Contacts for the Top 100 Facilities (Python)**

Using the `admin_details` dataset and the `facilities` dataset, create a Python script that matches **administrator contact information for the top 100 facilities by PBJ hours**.

### **Requirements**

1. **Identify the top 100 facilities by total PBJ hours**
   - Use the `pbj_hours` table.

2. **Match admin contacts for those 100 facilities**
   - Use the 'admin_details' table

3. **Final output must include one row per facility**  
   Columns:
   - `cms_id`
   - `facility_name`
   - `address`
   - `city`
   - `zip`
   - `admin_name`
   - `admin_email`

---

### **Goal**
Return **as many matched administrator contacts as possible** for the top 100 facilities.


In [ ]:
# QUESTION 2
# Write your Python code below.


# Your code here

run_query("""
WITH ChainAggregates AS (
    -- 1. Combine facility metadata with pbj_hours
    SELECT
        f.chain_name,
        f.cms_id,
        p.total_hours
    FROM facilities f
    JOIN pbj_hours p ON f.cms_id = p.cms_id
    WHERE f.chain_name IS NOT NULL
      AND f.chain_name != ''
      AND p.total_hours > 0
),
StatewideTotal AS (
    -- 2. Get the denominator for the statewide percentage
    SELECT SUM(total_hours) as state_total
    FROM pbj_hours
    WHERE total_hours > 0
)
SELECT
    ca.chain_name,
    -- Count distinct facilities in the chain from the metadata table
    (SELECT COUNT(DISTINCT cms_id) FROM facilities WHERE chain_name = ca.chain_name) AS total_facilities,
    -- Count distinct facilities that actually reported hours
    COUNT(DISTINCT ca.cms_id) AS facilities_with_pbj_hours,
    -- Sum of all agency hours (CNA + LPN + RN)
    SUM(ca.total_hours) AS total_agency_hours,
    -- Percentage of total statewide hours
    (SUM(ca.total_hours) * 100.0 / (SELECT state_total FROM StatewideTotal)) AS pct_of_statewide_hours
FROM ChainAggregates ca
GROUP BY ca.chain_name
ORDER BY total_agency_hours DESC
LIMIT 10
""")

,chain_name,total_facilities,facilities_with_pbj_hours,total_agency_hours,pct_of_statewide_hours
0,PACS GROUP,125,84,172739.87,14.984653
1,MARINER HEALTH CARE,18,17,160999.32,13.966196
2,ASPEN SKILLED HEALTHCARE,30,22,49993.21,4.336757
3,BRIUS MANAGEMENT,36,18,47877.09,4.153190
4,ROLLINS-NELSON HEALTHCARE MANAGEMENT,10,7,45537.63,3.950250
5,GOLDEN SNF OPERATIONS,7,7,44516.85,3.861700
6,THE ENSIGN GROUP,69,34,39047.36,3.387239
7,EVA CARE GROUP,9,8,31544.50,2.736388
8,PROMEDICA SENIOR CARE,6,5,17417.54,1.510918
9,LINKS HEALTHCARE GROUP,21,11,16445.25,1.426575


### 📌 **Question 3 — Deal Revenue & Commission Attribution (SQL)**

You are given a new table called `deals` that attributes facilities to one or two sales reps who signed them.
Each deal may be:

- fully attributed to one rep, or  
- split between two reps (**75% / 25%** or **50% / 50%**)


You will also use the `shifts` table, where each shift includes:

- `hours`  
- `pay_rate`  
- `charge_rate`

Revenue from a shift is calculated as:

revenue = (charge_rate - pay_rate) * hours


---

### **Requirements**

1. **Determine the 30-day revenue window for each deal**
   - For each `cms_id` in `deals`, find the **first shift date** in the `shifts` table.
   - Include all shifts occurring **within 30 days after** that first shift date.

2. **Calculate total revenue for each deal**
   - Use the revenue formula above.
   - Sum all revenue across all qualifying shifts for each deal.

3. **Allocate commissions to sales reps**
   - Each rep earns **20% of attributed revenue**, allocated by the deal’s split percentages:
     ```
     commission = revenue * 0.20 * rep_split_pct
     ```
   - `rep_split_pct` will be:
     - `1.0` for a fully owned deal  
     - `0.75` or `0.25` for 75/25 splits  
     - `0.50` for equal splits  

4. **Final output must return one row per rep per deal**  
   Columns:
   - `deal_id`  
   - `cms_id`  
   - `rep_name`  
   - `rep_split_pct`  
   - `total_revenue`  
   - `commission_owed`

5. **Sort the final output**
   - First by `commission_owed` **DESC**  
   - Then by `rep_name` **ASC**

---

### **Goal**

Produce a result showing how much **revenue** each deal generated and how much **commission** each rep earns based on the predefined attribution splits.



In [ ]:
# QUESTION 3
# Write your SQL query below
# Reminder that SQL must be wrapped in run_query() - example below

run_query("""
WITH FirstShift AS (
    -- 1. Find the first shift date for each facility (cms_id)
    SELECT
        cms_id,
        MIN(date) as first_date
    FROM shifts
    GROUP BY cms_id
),
QualifyingRevenue AS (
    -- 2. Sum revenue only for shifts within 30 days of the first shift
    -- Revenue = (charge_rate - pay_rate) * hours
    SELECT
        s.cms_id,
        SUM((s.charge_rate - s.pay_rate) * s.hours) AS total_revenue
    FROM shifts s
    JOIN FirstShift fs ON s.cms_id = fs.cms_id
    WHERE s.date BETWEEN fs.first_date AND date(fs.first_date, '+30 days')
    GROUP BY s.cms_id
),
RepSplits AS (
    -- 3. Unpivot reps so each gets a row with their percentage
    SELECT
        deal_id, cms_id,
        rep_primary AS rep_name,
        split_primary_pct AS rep_split_pct
    FROM deals
    UNION ALL
    SELECT
        deal_id, cms_id,
        rep_secondary AS rep_name,
        split_secondary_pct AS rep_split_pct
    FROM deals
    WHERE rep_secondary IS NOT NULL
)
-- 4. Final calculation and sorting
SELECT
    rs.deal_id,
    rs.cms_id,
    rs.rep_name,
    rs.rep_split_pct,
    qr.total_revenue,
    (qr.total_revenue * 0.20 * rs.rep_split_pct) AS commission_owed
FROM RepSplits rs
JOIN QualifyingRevenue qr ON rs.cms_id = qr.cms_id
ORDER BY commission_owed DESC, rep_name ASC
""")

,deal_id,cms_id,rep_name,rep_split_pct,total_revenue,commission_owed
0,8,555638,Morgan,1.00,4136,827.2
1,9,555923,Casey,1.00,3636,727.2
2,7,555200,Alex,1.00,3572,714.4
3,5,055008,Taylor,1.00,3500,700.0
4,4,055350,Jordan,1.00,3448,689.6
5,10,055462,Morgan,1.00,2604,520.8
6,2,056425,Casey,1.00,2540,508.0
7,3,555796,Taylor,0.75,2768,415.2
8,6,555116,Taylor,0.75,2408,361.2
9,1,056008,Jordan,1.00,1452,290.4


## 🔥 **Extra Credit**

You will not be penalized at all if you leave this section blank, but if you complete the first 3 questions with time to spare here is another couple you can work on to showcase additional skills.


### 🔥 Extra Credit #1 — Interactive Facility Map

Using the facilities table and the pbj_hours table, create an **interactive map** that visualizes all facilities in the dataset.

Your map should include:

- **One point per facility**
- **Dot size proportional to total PBJ hours** (RN + LPN + CNA)
- **Hover tooltip** that displays:
  - Facility name  
  - Address (city + zip is acceptable)
  - Total PBJ Hours

**Again:** This question is completely optional and will not impact your score for the main assessment. It is an opportunity to showcase your Python + visualization skills if you finish early.

In [ ]:
# Extra Credit #1
# Write your Python code below.


# Your code here
import plotly.express as px

# 1. Aggregate total PBJ hours per facility
map_data = pbj_long_df.groupby('cms_id')['total_hours'].sum().reset_index()

# 2. Merge with facilities_df to get lat/long and metadata
map_df = map_data.merge(facilities_df, on='cms_id', how='left')

# 3. Create a combined address field for the tooltip
map_df['hover_address'] = map_df['city'] + ", " + map_df['zip'].astype(str)

# 4. Create the interactive map
fig = px.scatter_mapbox(
    map_df,
    lat="latitude",
    lon="longitude",
    size="total_hours",
    color="total_hours",
    color_continuous_scale=px.colors.cyclical.IceFire,
    size_max=15,
    zoom=5,
    mapbox_style="carto-positron",
    hover_name="name",
    hover_data={
        "latitude": False,
        "longitude": False,
        "total_hours": ":.2f",
        "hover_address": True
    },
    labels={"total_hours": "Total PBJ Hours", "hover_address": "Address"},
    title="Facility PBJ Staffing Activity Map"
)

fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
fig.show()



### 🔥 Extra Credit #2 — Adult Care Home CSV Scraper

Using Python, scrape the **Adult Care Home CSV** from the NC Department of Health and Human Services website:

🔗 https://info.ncdhhs.gov/dhsr/acls/faclistings.html

Your task:

1. Programmatically locate the **Adult Care Home CSV** link on that page.
2. Download the CSV file.
3. Load it into a **pandas DataFrame**.
4. Display the first 10 rows.

**Again:** This question is completely optional and will not impact your score for the main assessment. It is an opportunity to showcase your Python skills if you finish early.


In [ ]:
# Extra Credit #2
# Write your Python code below.


# Your code here

import requests
from bs4 import BeautifulSoup
import pandas as pd
import io

# 1. Target the specific Facility Listings page
url = "https://info.ncdhhs.gov/dhsr/acls/faclistings.html"
response = requests.get(url)
soup = BeautifulSoup(response.content, 'html.parser')

# 2. Look for the link - specifically looking for the [XLSX] or text version
csv_link = None
for link in soup.find_all('a', href=True):
    link_text = link.text.lower()
    # Looking for 'adult care' and excluding PDF to get the data file
    if "adult care" in link_text and "pdf" not in link_text:
        csv_link = link['href']
        break

# 3. Fix relative URLs
if csv_link and not csv_link.startswith('http'):
    # The base for this specific subpage is /dhsr/acls/
    csv_link = "https://info.ncdhhs.gov/dhsr/acls/" + csv_link

if csv_link:
    print(f"Success! Found data file at: {csv_link}")

    # 4. Download and Load
    # Note: If it's an .xlsx, use pd.read_excel. If it's .txt/.csv, use read_csv.
    if csv_link.endswith('.xlsx'):
        df_adult_care = pd.read_excel(csv_link)
    else:
        csv_response = requests.get(csv_link)
        # Using sep=None with engine='python' lets pandas auto-detect
        # if the file is comma or tab delimited.
        df_adult_care = pd.read_csv(io.StringIO(csv_response.text), sep=None, engine='python', on_bad_lines='skip')

    # 5. Display
    print("\nFirst 10 Rows:")
    display(df_adult_care.head(10))
else:
    # Fallback: Directly try the known data path if scraping fails
    print("Scraper couldn't find the link, attempting direct path...")
    direct_url = "https://info.ncdhhs.gov/dhsr/data/ahlist.txt"
    df_adult_care = pd.read_csv(direct_url, sep=None, engine='python', on_bad_lines='skip')
    display(df_adult_care.head(10))


Success! Found data file at: https://info.ncdhhs.gov/dhsr/acls//dhsr/acls/index.html

First 10 Rows:


,<!DOCTYPE,html,PUBLIC,-//W3C//DTD XHTML 1.0 Strict//EN,http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd>
0,<html,"xmlns=""http://www.w3.org/1999/xhtml"">",None,None,NaN
1,<head>,None,None,None,NaN
2,<meta,"http-equiv=""Content-Type""","content=""text/html;","charset=iso-8859-1""/>",NaN
3,<style,"type=""text/css"">",None,None,NaN
4,<!--,None,None,None,NaN
5,"body{margin:0;font-size:.7em;font-family:Verdana,","Arial,","Helvetica,",sans-serif;background:#EEEEEE;},NaN
6,fieldset{padding:0,15px,10px,15px;},NaN
7,h1{font-size:2.4em;margin:0;color:#FFF;},None,None,None,NaN
8,h2{font-size:1.7em;margin:0;color:#CC0000;},NaN,None,None,NaN
9,h3{font-size:1.2em;margin:10px,0,0,0;color:#000000;},NaN
